[env: optonpx]

# 2.1 Concatenate several ephys data sections of a single session

Typically concatenate the behavior and the optotagging section of a single session into one
processed recording for spike sorting. If you have a different session structure to process,
it's also easy to make modifications based on this notebook.

## Steps

Per probe and per session, in order:

1. **Laser artifact mask** (optotagging session only): laser pulses are detected on the ADC
   (`src.detect_laser_pulses`), the ADC→probe lag is estimated (`src.estimate_artifact_lag`),
   and a short window around every light onset and offset is masked with
   `spikeinterface.preprocessing.remove_artifacts(mode="linear")`.
2. **Detect common-mode noise** on the laser-masked data.
3. **Common-mode noise mask**: each detected event is masked with `remove_artifacts(mode="linear")`.
4. **NP2.0 phase shift** (`spikeinterface.preprocessing.phase_shift`).

Then the sessions are **concatenated** (`concatenate_recordings`) and written as int16
(`write_binary_recording`), together with sidecar files. The Kilosort channel map is made by notebook 2.2.

Not done here (both are included in Kilosort):

1. filter (TODO)
2. common average reference (CAR)

**Raw recordings are never modified**: SpikeInterface opens them read-only and
all output goes under `processed_session`.

## Why each step

### Laser artifact mask

When the laser light hits the Neuropixels probe it produces a photoelectric artifact at light
onset and at light offset. It is large, appears on many channels at once, and is time-locked to
every pulse. If it is left in the data, Kilosort detects it as spikes: the sorting then contains
artifact units, and real units can pick up artifact "spikes" at a fixed latency after each pulse,
which looks exactly like a light response and corrupts the optotagging test.

The ADC laser signal and the probe artifact are not perfectly aligned, so the artifact lag is
estimated from the data first, on one probe channel:

- **Channel**: `preprocess.mask_laser_artifact.probe_channel` in the metadata if it is set; otherwise
  it is picked automatically (`src.select_artifact_channel`). The pulse-triggered average of ~50
  pulses is computed on every channel; a channel is reliable when it is not a bad channel, never
  saturates and its artifact is far above the pulse-to-pulse variability (t value ≥ 5). Among the
  reliable channels with the largest 25 % of artifact amplitudes, the one with the highest t value
  is used. The chosen channel, its source (metadata / auto), polarity, amplitude and t value are
  always printed, and the notebook stops if no channel is reliable.
- **Polarity** is read from the same average. The artifact is normally negative; a positive one
  raises a warning. The lag is then the median time of the artifact extremum (with that polarity)
  over up to 200 pulses. Light reaches every channel at the same time, so one channel is enough.
- The QC cell shows the lag histogram, the pulse-triggered average of the chosen channel with the
  mask window, and the artifact amplitude of every channel.

The mask
replaces [−`pre_on_ms`, +`post_on_ms`] around each lag-corrected onset and
[−`pre_off_ms`, +`post_off_ms`] around each offset with a straight line between the window edges.
The window is kept short so that few real spikes are lost.

### Common-mode noise mask

Common-mode noise is a transient that hits (almost) every channel at once, e.g. the animal's
mouth touching the reward spout combined with imperfect grounding.

- **Why Kilosort does not remove it**: Kilosort's CAR subtracts the median across channels at
  every sample, which only cancels noise that is identical on all channels. The amplitude and
  shape of these transients differ across channels (electrode position, impedance), so a residual
  remains after CAR. Kilosort's 300 Hz high-pass then turns this slow or step-like residual into a
  short, sharp waveform that looks like a spike, which can be detected and even clustered into a
  fake unit.
- **Detection**: a copy of the good channels is centred per channel (mean removed, this is *not*
  CAR) and high-passed at 300 Hz. CAR is deliberately not applied to this copy because it would
  remove the very common-mode signal we are looking for. Each channel's noise level is estimated
  with the MAD (median absolute deviation, from 100 random 0.5 s chunks with a fixed seed, so every
  run gives the same events), and its threshold is `threshold_mad` × MAD. A sample is a candidate
  sample when at least `min_fraction` of the good channels exceed their threshold at the same time
  (with `min_fraction` = 0.5 this means the median |z| over channels ≥ `threshold_mad`).
- **Sharpness**: many candidates are smooth common-mode waves (1–10 ms), which Kilosort's CAR does
  remove. Only sharp transients (rise within 1–2 samples, laser-artifact-like) survive the CAR, so a
  candidate is masked only when its slope — the median over channels of |z[t] − z[t−1]|, largest
  within ±0.5 ms — is at least `min_slope_mad` (4). On 2026-09-04 the two groups were clearly
  apart (smooth ≤ 3.7, sharp ≥ 4.7; every optotagging candidate was smooth); the QC cell plots the
  slope of every candidate so this can be checked for each session.
- **Why this separates noise from spikes**: a real spike is local and only reaches a few to a
  dozen neighbouring channels, so it never drives a large fraction of the probe over threshold at
  once, however large it is.
- **Mask**: each event is masked over [start − `pre_ms`, end + `post_ms`]; events whose windows
  overlap are merged. As for the laser artifact, the window is replaced with a straight line
  (`remove_artifacts(mode="linear")`). Every masked window is saved to `noise_events.csv` for
  later checks.
- Each probe is detected and masked independently, with its own thresholds and parameters.
- Detection runs on every session (behavior and optotagging), each with its own noise levels.
  The optotagging session should not contain this kind of noise: any event there raises a warning
  (with how many events are within 5 ms of a laser on/off edge, which would point to a laser mask
  that is too narrow or a wrong lag), and more than `max_optotagging_events` (10) events stop the
  notebook.
- **Check before writing**: SpikeInterface is lazy, so nothing is written until the write cell.
  Run up to the noise-mask QC cell first: it shows the slope histogram of all candidates with the
  `min_slope_mad` line, and for the largest, longest, weakest and a few random masked events it plots raw | after the mask | after the mask + phase shift + Kilosort-4-like preprocessing
  (mean removal, median CAR, 300 Hz high-pass), with the mask window shaded. The window is wide
  enough when the last column shows no spike-like residue next to the shaded area. Only then run
  the write cell.

### NP2.0 phase shift

Neuropixels probes do not sample all channels at the same moment: each ADC digitises a group of
channels one after another within one sample period. On NP2.0, channel `c` is sampled
`((c // 2) % 16) / 16` of a sample period after the nominal time, i.e. up to ~31 µs late at 30 kHz
(`src.preprocess.npx2_sample_shifts`). `phase_shift` delays every channel by its own fractional
amount in the frequency domain so that all channels are back on a common time grid.

This matters most when CAR is done before or during sorting (Kilosort's `do_CAR`): CAR assumes that
the same sample index means the same moment on every channel. Without the shift, fast signals that
are common to all channels are not aligned across channels and are only partly removed by the CAR.

### Concatenation

The behavior and the optotagging sessions are sorted together so that each unit has the same
identity in both: the optotagging session tells which behavior-session units are tagged.

- **Compatibility checks**: every session must have the same stream, channel count, sample rate
  and the same bank / shank / electrode for every channel (same probe configuration); otherwise
  the notebook stops.
- **Continuity checks**: within each session, `timestamps.npy` must have no gaps and match the
  `continuous.dat` size (a warning is printed for gaps).
- **Sample-based concatenation**: sessions are appended back to back in the order given by
  `preprocess.concatenate` in the metadata. Wall-clock time between sessions is ignored, so the
  concatenated time axis is a sample index, not Open Ephys time.
- **Output**: `<processed_session>/<probe>/continuous.dat` (int16) plus
  - `session_boundaries.json`: `sample_start` / `sample_end` of every session on the
    concatenated axis, used to split the sorting back per session;
  - `laser_events.csv`: every laser pulse as a concatenated-axis sample index (lag-corrected);
  - `noise_events.csv`: every masked noise window, both session-local and concatenated-axis;
  - `preprocessing.json`: all preprocessing parameters, per probe;
  - `<session_name>.yaml`: a copy of the metadata file as it was read when this notebook ran.

## Why this order

laser mask → detect common-mode noise → noise mask → phase shift → concatenate

1. **Laser mask before noise detection**: the laser artifact also hits many channels at the same
   time, so it would be detected as common-mode noise. Masking it first keeps the two kinds of
   artifact separate (each with its own window) and keeps `noise_events.csv` limited to real
   common-mode noise.
2. **Both masks before the phase shift**: the phase shift is a fractional-sample delay done in the
   frequency domain, which is equivalent to interpolating each channel with a sinc kernel. Applied
   to a sharp, large-amplitude edge, it produces ringing: oscillating tails that spread in time
   beyond the narrow mask window, with a different shape on each channel (each channel has its own
   shift). Masking first gives the phase shift a continuous signal (the linear mask connects the
   window edges), so no ringing is created.
3. **Phase shift per session, before concatenation**: the junction between two sessions is also a
   discontinuity; shifting each session on its own means an FFT block never crosses a session
   boundary.

In [ ]:
from pathlib import Path

import yaml

# ========== Specify the session to preprocess ==========
session_name = "IM-1979_2026-09-11"
base_dir_raw = Path(r"E:/")
base_dir_output = Path(r"E:/")
is_overwrite = False
processed_suffix = "_noisemask"   # output folder = <sessions.processed><suffix>; keeps the plain 2.2 output
# ========================================================

# Load json file
metadata_dir = Path.cwd().parent
metadata_path = Path(metadata_dir / "metadata" / f"{session_name}.yaml")
print(metadata_path)
metadata_text = metadata_path.read_text(encoding="utf-8")   # copied as-is into the processed folder
config = yaml.safe_load(metadata_text)

base_dir_raw = base_dir_raw / config["animal"]
behavior_session = base_dir_raw / "ephys" / config["sessions"]["behavior"]
tagging_session = base_dir_raw / "ephys" / config["sessions"]["optotagging"]
processed_session = base_dir_output / config["animal"] / "ephys" / (config["sessions"]["processed"] + processed_suffix)

probe_names = config["streams"]["probes"]
n_probe = len(probe_names)
print(f"There are {len(probe_names)} probes: {', '.join(probe_names)}")

if processed_session.exists() and not is_overwrite:
    raise ValueError("Processed folder already existed and 'overwrite' is False.")

In [ ]:
import csv
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append('..')
import spikeinterface.full as si
from spikeinterface.core import concatenate_recordings, write_binary_recording
from spikeinterface.core.job_tools import ChunkRecordingExecutor
from spikeinterface.preprocessing import phase_shift, remove_artifacts

from src import (
    detect_laser_pulses,
    estimate_artifact_lag,
    oe_load_adc_column,
    oe_parse_folders,
    oe_parse_params,
    select_artifact_channel,
)
from src.preprocess import (
    DEFAULT_ARTIFACT_MASK,
    DEFAULT_LAG_ESTIMATION,
    DEFAULT_PULSE_DETECTION,
    npx2_sample_shifts,
)

print("spikeinterface", si.__version__)

# Per-probe parameter blocks from the metadata (defaults filled in from src.preprocess)
default_phase_shift_params = {"apply": True}                       # + margin_ms from metadata
default_mask_laser_params  = {**DEFAULT_ARTIFACT_MASK, "apply": True,
                              "probe_channel": None}               # None -> pick the artifact channel automatically
default_car_params         = {"apply": False}                      # KS4 does CAR, per shank
default_mask_noise_params  = {"apply": True, "threshold_mad": 8.0, "min_fraction": 0.5,
                              "min_slope_mad": 4.0,               # sharpness: keeps artifacts, drops smooth common-mode waves
                              "pre_ms": 0.5, "post_ms": 1.0,
                              "max_optotagging_events": 10}       # more events in the optotagging session -> error

concatenate_order  = {}
phase_shift_params = {}
mask_laser_params  = {}
car_params         = {}
mask_noise_params  = {}
for probe_name in probe_names:
    preprocess_cfg = config[probe_name]["preprocess"]
    concatenate_order[probe_name]  = preprocess_cfg["concatenate"]
    phase_shift_params[probe_name] = {**default_phase_shift_params, **preprocess_cfg["phase_shift"]}
    mask_laser_params[probe_name]  = {**default_mask_laser_params,  **preprocess_cfg["mask_laser_artifact"]}
    car_params[probe_name]         = {**default_car_params,         **preprocess_cfg["car"]}
    mask_noise_params[probe_name]  = {**default_mask_noise_params,
                                      **preprocess_cfg.get("mask_noise_artifact", {})}

In [ ]:
# --- resolve session folders, parse Open Ephys, check they can be concatenated ---
session_dirs  = {"behavior": behavior_session, "optotagging": tagging_session}
session_roles = list(session_dirs)                           # ["behavior", "optotagging"]


def match_one_stream(items, stream_name, name_of):
    """Return the single item whose name ends with `stream_name`, case-insensitively
    (the metadata says 'probeA', Open Ephys says 'ProbeA')."""
    matches = [item for item in items
               if str(name_of(item)).split(".")[-1].lower() == stream_name.lower()]
    if len(matches) != 1:
        raise ValueError(f"probe {stream_name!r}: expected 1 match, got {[name_of(i) for i in items]}")
    return matches[0]


sessions = {}   # role -> {folder, adc_dir, adc_params, probes: {probe_name: {...}}}
for role, folder in session_dirs.items():
    _, oe_paths = oe_parse_folders(folder)
    probe_params_list, adc_params = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])
    session_info = {"folder": folder, "adc_dir": oe_paths["adc_stream"],
                    "adc_params": adc_params, "probes": {}}
    for probe_name in probe_names:
        probe_params = match_one_stream(probe_params_list, probe_name, lambda p: p["stream_name"])
        stream_dir   = match_one_stream(oe_paths["ephys_streams"], probe_name, lambda p: p.name)
        n_samples    = int(len(np.load(stream_dir / "timestamps.npy", mmap_mode="r")))
        session_info["probes"][probe_name] = {
            "params": probe_params, "stream_dir": stream_dir,
            "n_chan": int(probe_params["channel_count"]),
            "fs": float(probe_params["sample_rate"]), "n_samples": n_samples}
    sessions[role] = session_info

# concatenating only makes sense if every session used the exact same probe wiring
reference_role = session_roles[0]
for probe_name in probe_names:
    reference_params = sessions[reference_role]["probes"][probe_name]["params"]
    for role in session_roles[1:]:
        params = sessions[role]["probes"][probe_name]["params"]
        for field in ("stream_name", "channel_count", "sample_rate"):
            if str(params[field]) != str(reference_params[field]):
                raise ValueError(f"{probe_name}: {field} differs "
                                 f"({reference_role}={reference_params[field]!r} vs {role}={params[field]!r})")
        for field in ("channel_bank", "channel_shank", "channel_electrode"):
            for channel in range(int(reference_params["channel_count"])):
                if reference_params[field].get(channel) != params[field].get(channel):
                    raise ValueError(f"{probe_name} ch{channel} {field} differs between "
                                     f"{reference_role} and {role} — different probe/bank config, "
                                     "cannot concatenate")

print("compatibility OK: stream / channel_count / sample_rate / per-channel bank+shank+electrode match")
for probe_name in probe_names:
    sample_counts = [sessions[role]["probes"][probe_name]["n_samples"] for role in session_roles]
    fs = sessions[reference_role]["probes"][probe_name]["fs"]
    print(f"  {probe_name}: {sample_counts} -> {sum(sample_counts)} samples ({sum(sample_counts) / fs:.1f} s)")

In [ ]:
# --- safety: the processed output must not sit inside any raw recording folder ---
output_folder = processed_session.resolve()
for role, folder in session_dirs.items():
    raw_folder = folder.resolve()
    if (output_folder == raw_folder
            or raw_folder in output_folder.parents
            or output_folder in raw_folder.parents):
        raise RuntimeError(f"processed_session\n  {output_folder}\noverlaps raw recording ({role})\n"
                           f"  {raw_folder}\nRefusing to run — raw data must stay untouched.")
print("processed_session is clear of every raw recording folder — OK")

In [ ]:
# --- continuity checks: no gaps within a session, dat matches timestamps ---
for role in session_roles:
    for probe_name in probe_names:
        probe_info = sessions[role]["probes"][probe_name]
        timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy")
        intervals = np.diff(timestamps)
        expected_interval = 1.0 / probe_info["fs"]
        n_gap = int(np.count_nonzero(np.abs(intervals - expected_interval) > 0.5 * expected_interval))

        dat_bytes = (probe_info["stream_dir"] / "continuous.dat").stat().st_size
        bytes_per_sample = probe_info["n_chan"] * 2
        assert dat_bytes % bytes_per_sample == 0, \
            f"{probe_info['stream_dir']}: dat size not a multiple of n_chan*2"
        assert dat_bytes // bytes_per_sample == probe_info["n_samples"], \
            f"{probe_info['stream_dir']}: dat has {dat_bytes // bytes_per_sample} samples, " \
            f"timestamps has {probe_info['n_samples']}"

        max_deviation_us = np.abs(intervals - expected_interval).max() * 1e6
        print(f"{role:12s} {probe_name}: {probe_info['n_samples']} samp, {n_gap} gaps >50% "
              f"(max dev {max_deviation_us:.1f} us), dat size OK")
        if n_gap:
            print("  WARNING: sample gaps — session may have been paused/dropped; "
                  "concatenation assumes each session is gap-free")

In [ ]:
# --- optotagging session: detect laser pulses, pick the artifact channel, estimate ADC->probe lag ---
laser_channels = {name: channel for name, channel in config["adc"]["channels"].items()
                  if name.endswith("_laser")}                # {"blue_laser": 1, "red_laser": 3}

# pulse detection: threshold from the metadata, everything else from the pipeline defaults
pulse_detection_cfg = {**DEFAULT_PULSE_DETECTION,
                       "threshold_volts": config["adc"]["threshold_volts"]}
lag_estimation_cfg = DEFAULT_LAG_ESTIMATION   # search_window_ms / max_pulses (channel + sign are picked below)

optotagging = sessions["optotagging"]
opto_results = {}   # (probe_name, laser_name) -> {onset_times, offset_times, lag_ms, per_pulse_ms, adc_channel, artifact}
for probe_name in probe_names:
    probe_info = optotagging["probes"][probe_name]
    probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
    bit_volts = float(probe_info["params"]["channel_bit_volts"])
    bad_channels = config[probe_name]["kilosort_params"]["bad_channels"]
    metadata_channel = mask_laser_params[probe_name]["probe_channel"]
    for laser_name, adc_channel in laser_channels.items():
        adc_volts, adc_timestamps, _ = oe_load_adc_column(
            optotagging["adc_dir"], optotagging["adc_params"], adc_channel)
        onset_times, offset_times = detect_laser_pulses(
            adc_volts, adc_timestamps,
            threshold_volts=pulse_detection_cfg["threshold_volts"],
            polarity=pulse_detection_cfg["polarity"],
            min_width_ms=pulse_detection_cfg["min_width_ms"],
            max_width_ms=pulse_detection_cfg["max_width_ms"],
            min_interpulse_ms=pulse_detection_cfg["min_interpulse_ms"])
        if len(onset_times) == 0:
            raise RuntimeError(f"{probe_name} {laser_name}: no pulses on ADC{adc_channel} "
                               "— check threshold / polarity / width")

        # channel + polarity for the lag: from the metadata if given, otherwise the channel with
        # a large and reliable pulse-triggered average (see select_artifact_channel)
        artifact = select_artifact_channel(
            probe_info["stream_dir"] / "continuous.dat", probe_timestamps,
            probe_info["n_chan"], onset_times, bad_channels=bad_channels,
            search_window_ms=tuple(lag_estimation_cfg["search_window_ms"]),
            probe_channel=metadata_channel)
        print(f"{probe_name} {laser_name:10s}: artifact channel {artifact['channel']} "
              f"({'from metadata' if artifact['source'] == 'metadata' else 'auto-selected'}), "
              f"{artifact['artifact_sign']}, {artifact['amplitude'] * bit_volts:.0f} uV, "
              f"t = {artifact['t_value']:.1f}, peak {artifact['peak_ms']:+.2f} ms "
              f"({artifact['eligible_mask'].sum()} channels with a reliable artifact)")
        if artifact["artifact_sign"] == "positive":
            warnings.warn(f"{probe_name} {laser_name}: the laser artifact on channel {artifact['channel']} "
                          "is POSITIVE (usually negative) — check the pulse-triggered average below")
        if artifact["source"] == "metadata" and not artifact["eligible"]:
            warnings.warn(f"{probe_name} {laser_name}: metadata channel {artifact['channel']} is bad, "
                          f"saturated or has no reliable artifact (t = {artifact['t_value']:.1f})")

        lag_ms, per_pulse_ms = estimate_artifact_lag(
            probe_info["stream_dir"] / "continuous.dat", probe_timestamps,
            probe_info["n_chan"], onset_times,
            probe_channel=artifact["channel"],
            search_window_ms=tuple(lag_estimation_cfg["search_window_ms"]),
            artifact_sign=artifact["artifact_sign"],
            max_pulses=lag_estimation_cfg["max_pulses"])
        lag_mad_ms = float(np.nanmedian(np.abs(per_pulse_ms - np.nanmedian(per_pulse_ms))))
        print(f"{probe_name} {laser_name:10s}: {len(onset_times):4d} pulses, "
              f"width {np.median((offset_times - onset_times) * 1e3):.1f} ms, "
              f"lag {lag_ms:+.3f} ms (MAD {lag_mad_ms:.3f})")
        opto_results[(probe_name, laser_name)] = {
            "onset_times": onset_times, "offset_times": offset_times, "lag_ms": lag_ms,
            "per_pulse_ms": per_pulse_ms, "adc_channel": adc_channel, "artifact": artifact}


In [ ]:
# --- QC: lag histogram + pulse-triggered average with the mask window overlaid ---
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

for (probe_name, laser_name), result in opto_results.items():
    probe_info = sessions["optotagging"]["probes"][probe_name]
    fs = probe_info["fs"]
    artifact = result["artifact"]
    probe_channel = artifact["channel"]
    mask_cfg = mask_laser_params[probe_name]
    probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
    source = np.memmap(probe_info["stream_dir"] / "continuous.dat", dtype="int16",
                       mode="r").reshape(probe_info["n_samples"], probe_info["n_chan"])

    n_before, n_after = int(0.005 * fs), int(0.015 * fs)
    snippets = []
    for onset_time in result["onset_times"][:200]:
        center = int(np.searchsorted(probe_timestamps, onset_time))
        if n_before <= center < probe_info["n_samples"] - n_after:
            snippets.append(source[center - n_before:center + n_after, probe_channel].astype("float64"))
    snippets = np.array(snippets)
    t_ms = (np.arange(-n_before, n_after) / fs) * 1e3

    fig, axes = plt.subplots(1, 3, figsize=(16, 3.2))
    valid_lags = result["per_pulse_ms"][~np.isnan(result["per_pulse_ms"])]
    axes[0].hist(valid_lags, 40, color="0.6")
    axes[0].axvline(result["lag_ms"], color="r")
    axes[0].set(xlabel="artifact lag (ms)", ylabel="pulses",
                title=f"{probe_name} {laser_name}: lag {result['lag_ms']:+.3f} ms")
    if len(snippets):
        mean_trace = snippets.mean(0)
        axes[1].plot(t_ms, mean_trace - np.median(mean_trace), "k")
    axes[1].axvline(0, color="b", lw=0.8, label="ADC onset")
    axes[1].axvline(result["lag_ms"], color="r", lw=0.8, label="est. lag")
    axes[1].axvspan(result["lag_ms"] - mask_cfg["pre_on_ms"],
                    result["lag_ms"] + mask_cfg["post_on_ms"],
                    color="r", alpha=0.15, label="onset mask")
    axes[1].set(xlabel="time from ADC onset (ms)", ylabel="uV (rel.)",
                title=f"{probe_name} ch{probe_channel} pulse-triggered avg (n={len(snippets)})")
    axes[1].legend(fontsize=7)
    # artifact amplitude on every channel; the chosen channel in red
    bit_volts = float(probe_info["params"]["channel_bit_volts"])
    channels = np.arange(len(artifact["amplitudes"]))
    eligible = artifact["eligible_mask"]
    axes[2].plot(channels[eligible], artifact["amplitudes"][eligible] * bit_volts, ".", ms=3,
                 color="0.4", label="reliable")
    axes[2].plot(channels[~eligible], artifact["amplitudes"][~eligible] * bit_volts, "x", ms=4,
                 color="0.7", label="bad / saturated / unreliable")
    axes[2].plot(probe_channel, artifact["amplitude"] * bit_volts, "o", mfc="none", color="r",
                 label=f"ch{probe_channel} ({artifact['source']})")
    axes[2].set(xlabel="channel", ylabel="artifact amplitude (uV)",
                title=f"pulse-triggered average, n={artifact['n_pulses']} pulses")
    axes[2].legend(fontsize=7)
    fig.tight_layout()
    plt.show()

In [ ]:
# --- per-session recording up to the laser mask, common-mode noise detection and mask ---
def laser_edge_frames(probe_name):
    """Lag-corrected laser on / off times of every laser as optotagging-session sample indices."""
    probe_info = sessions["optotagging"]["probes"][probe_name]
    probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
    onset_frames, offset_frames = [], []
    for laser_name in laser_channels:
        result = opto_results[(probe_name, laser_name)]
        lag_seconds = result["lag_ms"] / 1000.0
        onset_frames.append(np.searchsorted(probe_timestamps, result["onset_times"] + lag_seconds))
        offset_frames.append(np.searchsorted(probe_timestamps, result["offset_times"] + lag_seconds))
    last_sample = probe_info["n_samples"] - 2
    return (np.clip(np.concatenate(onset_frames), 1, last_sample),
            np.clip(np.concatenate(offset_frames), 1, last_sample))


def build_session_recording(probe_name, role):
    """Raw int16 recording of one session (read-only), laser artifacts masked for optotagging.
    Shared by the noise detection below and the write cell, so both see the same data."""
    probe_info = sessions[role]["probes"][probe_name]
    mask_cfg = mask_laser_params[probe_name]
    recording = si.read_binary(probe_info["stream_dir"] / "continuous.dat",
                               sampling_frequency=probe_info["fs"], dtype="int16",
                               num_channels=probe_info["n_chan"])
    n_laser_pulses = 0
    if role == "optotagging" and mask_cfg["apply"]:
        onset_frames, offset_frames = laser_edge_frames(probe_name)
        # remove_artifacts takes one (ms_before, ms_after) per call -> onset and offset separately
        recording = remove_artifacts(recording, [onset_frames],
                                     ms_before=mask_cfg["pre_on_ms"],
                                     ms_after=mask_cfg["post_on_ms"], mode="linear")
        recording = remove_artifacts(recording, [offset_frames],
                                     ms_before=mask_cfg["pre_off_ms"],
                                     ms_after=mask_cfg["post_off_ms"], mode="linear")
        n_laser_pulses = len(onset_frames)
    return recording, n_laser_pulses


NOISE_EVENT_COLUMNS = ["start_sample", "end_sample", "peak_sample", "peak_fraction", "amp_mad",
                       "slope_mad", "duration_ms", "mask_start", "mask_end"]
SLOPE_WINDOW_MS = 0.5   # the slope of an event is the largest one within +-0.5 ms of its samples


def _fraction_init(recording, mad, threshold_mad, min_fraction, slope_half):
    return {"recording": recording, "mad": mad, "threshold_mad": threshold_mad,
            "min_fraction": min_fraction, "slope_half": slope_half}


def _fraction_chunk(segment_index, start_frame, end_frame, worker_dict):
    """Event samples of one chunk, with their channel fraction, amplitude and slope (all in MADs)."""
    recording, half = worker_dict["recording"], worker_dict["slope_half"]
    # read slope_half (+1 for the first difference) extra samples on each side
    a = max(start_frame - half - 1, 0)
    b = min(end_frame + half, recording.get_num_samples(segment_index=segment_index))
    z = recording.get_traces(segment_index=segment_index, start_frame=a, end_frame=b) / worker_dict["mad"]
    inner = slice(start_frame - a, end_frame - a)
    fraction = (np.abs(z[inner]) > worker_dict["threshold_mad"]).mean(axis=1)
    hits = np.flatnonzero(fraction >= worker_dict["min_fraction"]) + (start_frame - a)   # rows of z
    if len(hits) == 0:
        empty = np.zeros(0)
        return np.zeros(0, np.int64), empty, empty, empty
    amp = np.median(np.abs(z[hits]), axis=1)
    # slope[t] = median over channels |z[t] - z[t-1]|, only on rows within slope_half of an event sample
    rows = np.unique(np.clip((hits[:, None] + np.arange(-half, half + 1)).ravel(), 1, len(z) - 1))
    slope = np.zeros(len(z))
    slope[rows] = np.median(np.abs(z[rows] - z[rows - 1]), axis=1)
    padded = np.pad(slope, half)
    slope_near = np.lib.stride_tricks.sliding_window_view(padded, 2 * half + 1)[hits].max(axis=1)
    return a + hits, fraction[hits - (start_frame - a)], amp, slope_near


def detect_common_mode_events(recording, good_channel_ids, threshold_mad, min_fraction,
                              pre_ms, post_ms, job_kwargs):
    """Candidate events: samples where >= min_fraction of the good channels exceed threshold_mad MADs
    at once, on a 300 Hz high-passed, NOT CARed copy (CAR would remove exactly what we look for).
    Each event also gets its amplitude (`amp_mad`: largest median |z| over channels) and sharpness
    (`slope_mad`: largest median over channels of |z[t] - z[t-1]|, within +-0.5 ms). The caller keeps
    the sharp ones (slope_mad >= min_slope_mad). Returns one row per event in session-local samples,
    with its mask window [mask_start, mask_end]; events whose windows would overlap are merged."""
    fs = recording.get_sampling_frequency()
    n_samples = recording.get_num_samples()
    detect = si.highpass_filter(si.center(recording.select_channels(good_channel_ids), mode="mean", seed=0),
                                freq_min=300.0, filter_order=3)
    # fixed seed + 100 x 500 ms chunks -> the same noise levels (and events) on every run
    mad = si.get_noise_levels(detect, method="mad", return_in_uV=False,
                              random_slices_kwargs={"seed": 0, "num_chunks_per_segment": 100})
    flat = [cid for cid, m in zip(good_channel_ids, mad) if m == 0]
    if flat:
        print(f"  WARNING: flat channels (MAD = 0) left out of the detection; add them to bad_channels: {[int(cid) for cid in flat]}")
        good_channel_ids = [cid for cid, m in zip(good_channel_ids, mad) if m > 0]
        detect = detect.select_channels(good_channel_ids)
        mad = mad[mad > 0]
    slope_half = int(round(SLOPE_WINDOW_MS / 1000 * fs))
    executor = ChunkRecordingExecutor(detect, _fraction_chunk, _fraction_init,
                                      (detect, mad, threshold_mad, min_fraction, slope_half),
                                      handle_returns=True, job_name="noise detection", **job_kwargs)
    results = executor.run()
    hits = np.concatenate([r[0] for r in results]).astype(np.int64)
    fractions, amps, slopes = (np.concatenate([r[k] for r in results]) for k in (1, 2, 3))
    order = np.argsort(hits)
    hits, fractions, amps, slopes = hits[order], fractions[order], amps[order], slopes[order]

    pre = int(round(pre_ms / 1000 * fs))
    post = int(round(post_ms / 1000 * fs))
    rows = []
    if len(hits):
        # windows [start - pre, end + post] of neighbouring events overlap when the gap <= pre + post
        breaks = np.flatnonzero(np.diff(hits) > pre + post) + 1
        for run, frac, amp, slope in zip(np.split(hits, breaks), np.split(fractions, breaks),
                                         np.split(amps, breaks), np.split(slopes, breaks)):
            rows.append({"start_sample": int(run[0]), "end_sample": int(run[-1]),
                         "peak_sample": int(run[np.argmax(amp)]), "peak_fraction": float(frac.max()),
                         "amp_mad": float(amp.max()), "slope_mad": float(slope.max())})
    events = pd.DataFrame(rows, columns=NOISE_EVENT_COLUMNS[:6])
    events["duration_ms"] = (events["end_sample"] - events["start_sample"] + 1) / fs * 1000
    events["mask_start"] = (events["start_sample"] - pre).clip(lower=1)
    events["mask_end"] = (events["end_sample"] + post).clip(upper=n_samples - 2)
    return events


def apply_noise_mask(recording, events):
    """Linear-interpolate exactly the samples [mask_start, mask_end] on all channels.
    remove_artifacts takes one ms_after per call and replaces trigger .. trigger + int(ms_after * fs / 1000)
    (inclusive), so windows are grouped by their exact length in samples."""
    if len(events) == 0:
        return recording
    fs = recording.get_sampling_frequency()
    span = events["mask_end"] - events["mask_start"]          # samples after the trigger
    for n_after, group in events.groupby(span):
        recording = remove_artifacts(recording, [group["mask_start"].to_numpy()], ms_before=0.0,
                                     ms_after=(int(n_after) + 0.5) / fs * 1000, mode="linear")
    return recording

In [ ]:
# --- detect common-mode noise per probe x session (optotagging: after the laser mask) ---
# reads every session once more; set n_jobs to 1 if multithreading misbehaves
detect_job_kwargs = {"n_jobs": 4, "chunk_duration": "2s", "progress_bar": True}

near_laser_ms = 5.0   # an optotagging event this close to a laser on/off edge is reported as laser-related

noise_candidates = {}   # (probe_name, role) -> every candidate (amplitude criterion only), for the QC
noise_events = {}       # (probe_name, role) -> sharp events only (these are masked), session-local samples
too_many_events = []
for probe_name in probe_names:
    noise_cfg = mask_noise_params[probe_name]
    bad_channels = set(config[probe_name]["kilosort_params"]["bad_channels"])
    for role in concatenate_order[probe_name]:
        if not noise_cfg["apply"]:
            noise_candidates[(probe_name, role)] = pd.DataFrame(columns=NOISE_EVENT_COLUMNS)
            noise_events[(probe_name, role)] = pd.DataFrame(columns=NOISE_EVENT_COLUMNS)
            continue
        recording, _ = build_session_recording(probe_name, role)
        good_channel_ids = [cid for i, cid in enumerate(recording.channel_ids) if i not in bad_channels]
        print(f"{probe_name} {role}:")
        candidates = detect_common_mode_events(
            recording, good_channel_ids, threshold_mad=noise_cfg["threshold_mad"],
            min_fraction=noise_cfg["min_fraction"], pre_ms=noise_cfg["pre_ms"],
            post_ms=noise_cfg["post_ms"], job_kwargs=detect_job_kwargs)
        # smooth common-mode waves are removed by Kilosort's CAR; only sharp transients are masked
        events = candidates[candidates["slope_mad"] >= noise_cfg["min_slope_mad"]].reset_index(drop=True)
        noise_candidates[(probe_name, role)] = candidates
        noise_events[(probe_name, role)] = events

        fs = recording.get_sampling_frequency()
        duration_s = recording.get_num_samples() / fs
        masked_s = float((events["mask_end"] - events["mask_start"] + 1).sum() / fs)
        print(f"  {len(candidates)} candidates >= {noise_cfg['threshold_mad']} MAD, "
              f"{len(events)} sharp (slope >= {noise_cfg['min_slope_mad']}) -> masked "
              f"({len(events) / duration_s:.3f} /s), "
              + (f"duration median {events['duration_ms'].median():.2f} / max {events['duration_ms'].max():.2f} ms, "
                 if len(events) else "")
              + f"masked {masked_s:.2f} s = {100 * masked_s / duration_s:.4f} % of the session")

        # the optotagging session should not contain common-mode noise -> warn, error above the limit
        if role == "optotagging" and len(events):
            edges = np.sort(np.concatenate(laser_edge_frames(probe_name)))
            nearest = np.searchsorted(edges, events["peak_sample"].to_numpy())
            distance = np.minimum(np.abs(events["peak_sample"].to_numpy() - edges[np.clip(nearest - 1, 0, len(edges) - 1)]),
                                  np.abs(edges[np.clip(nearest, 0, len(edges) - 1)] - events["peak_sample"].to_numpy()))
            n_near = int((distance / fs * 1000 <= near_laser_ms).sum())
            warnings.warn(f"{probe_name} optotagging: {len(events)} common-mode noise events "
                          f"({n_near} within {near_laser_ms} ms of a laser on/off edge"
                          + (" — mostly laser-related: check the laser lag / mask window" if n_near > len(events) / 2 else "")
                          + ")")
            if len(events) > noise_cfg["max_optotagging_events"]:
                too_many_events.append(f"{probe_name}: {len(events)} events "
                                       f"(limit {noise_cfg['max_optotagging_events']})")

if too_many_events:
    raise RuntimeError("too many common-mode noise events in the optotagging session — check the QC below "
                       "before writing:\n  " + "\n  ".join(too_many_events))


In [ ]:
# --- QC: is the noise mask window wide enough?  (nothing is written; run the next cell only if this looks right) ---
# Per probe x session: slope of every candidate (the masked events must stand apart from the smooth
# common-mode waves; if the two groups touch, check min_slope_mad), event statistics, then a few
# masked events (largest, longest, weakest, random), each as
# raw | after the noise mask | after the mask + phase shift + Kilosort-4-like preprocessing
# (mean removal, median CAR over the good channels, 300 Hz high-pass). The mask window is shaded.
# The window is wide enough when the last column shows no spike-like residue next to the shaded area.
%matplotlib inline
n_qc_random = 2            # random events per probe x session, on top of largest / longest / weakest
n_qc_channels = 8          # traces per panel, evenly spaced over the good channels
qc_window_ms = (-10, 20)   # around the event start
rng = np.random.default_rng(0)


def pick_qc_events(events):
    """{label: row index}: largest (amplitude), longest, weakest (smallest amplitude), random."""
    picks = {"largest": int(events["amp_mad"].idxmax()),
             "longest": int(events["duration_ms"].idxmax()),
             "weakest": int(events["amp_mad"].idxmin())}
    rest = [i for i in events.index if i not in picks.values()]
    for i in rng.choice(rest, size=min(n_qc_random, len(rest)), replace=False):
        picks[f"random {int(i)}"] = int(i)
    labels = {}
    for label, i in picks.items():                     # one figure per event, all labels in the title
        labels[i] = f"{labels[i]} / {label}" if i in labels else label
    return labels


def plot_stacked(ax, t_ms, x, spacing, title):
    for k in range(x.shape[1]):
        ax.plot(t_ms, x[:, k] - k * spacing, color="k", lw=0.6)
    ax.set(title=title, xlabel="time from event start (ms)", yticks=[])
    ax.set_xlim(t_ms[0], t_ms[-1])


for (probe_name, role), events in noise_events.items():
    probe_info = sessions[role]["probes"][probe_name]
    fs = probe_info["fs"]
    candidates = noise_candidates[(probe_name, role)]
    min_slope = mask_noise_params[probe_name]["min_slope_mad"]
    if len(candidates) == 0:
        print(f"{probe_name} {role}: no candidates")
        continue

    fig, axes = plt.subplots(1, 4, figsize=(18, 2.8), gridspec_kw={"width_ratios": [1.2, 1.6, 1, 1]})
    slope_bins = np.logspace(np.log10(max(candidates["slope_mad"].min(), 0.1)),
                             np.log10(max(candidates["slope_mad"].max(), min_slope) * 1.1), 50)
    axes[0].hist(candidates["slope_mad"], bins=slope_bins, color="0.5")
    axes[0].axvline(min_slope, color="r", ls="--")
    axes[0].set(xscale="log", yscale="log", xlabel="candidate slope (MAD / sample)", ylabel="candidates",
                title=f"{probe_name} {role}: {len(events)} of {len(candidates)} masked")
    if len(events):
        axes[1].hist(events["start_sample"] / fs, bins=200, color="0.4")
    axes[1].set(xlabel="session time (s)", ylabel="masked events")
    if len(events) > 1:
        axes[2].hist(np.diff(events["start_sample"]) / fs * 1000, bins=np.logspace(0, 6, 60), color="0.5")
        axes[2].set_xscale("log")
    axes[2].set(xlabel="inter-event interval (ms)")
    if len(events):
        axes[3].hist(events["duration_ms"], bins=40, color="0.5")
    axes[3].set(xlabel="event duration (ms)")
    fig.tight_layout()
    plt.show()
    if len(events) == 0:
        print(f"{probe_name} {role}: no masked events")
        continue

    bad_channels = set(config[probe_name]["kilosort_params"]["bad_channels"])
    before, _ = build_session_recording(probe_name, role)
    after = apply_noise_mask(before, events)
    good_ids = [cid for i, cid in enumerate(before.channel_ids) if i not in bad_channels]
    shown_ids = [good_ids[i] for i in np.linspace(0, len(good_ids) - 1, n_qc_channels).round().astype(int)]
    sorted_view = after
    if phase_shift_params[probe_name]["apply"]:
        sorted_view = phase_shift(sorted_view, margin_ms=phase_shift_params[probe_name].get("margin_ms", 40.0),
                                  inter_sample_shift=npx2_sample_shifts(probe_info["n_chan"]))
    sorted_view = si.highpass_filter(
        si.common_reference(si.center(sorted_view.select_channels(good_ids), mode="mean"),
                            reference="global", operator="median"),
        freq_min=300.0, filter_order=3)
    bit_volts = float(probe_info["params"]["channel_bit_volts"])

    for row, label in pick_qc_events(events).items():
        event = events.loc[row]
        a = int(event["start_sample"] + qc_window_ms[0] / 1000 * fs)
        b = int(event["start_sample"] + qc_window_ms[1] / 1000 * fs)
        if a < 0 or b > probe_info["n_samples"]:
            continue
        t_ms = (np.arange(a, b) - event["start_sample"]) / fs * 1000
        columns = {}
        for name, rec in (("raw", before), ("after noise mask", after), ("mask + KS4-like preprocessing", sorted_view)):
            x = rec.get_traces(start_frame=a, end_frame=b, channel_ids=shown_ids).astype(np.float32) * bit_volts
            columns[name] = x - np.median(x, axis=0, keepdims=True)
        # raw and masked share one spacing (direct comparison); the preprocessed column gets its own
        raw_spacing = 1.2 * float(np.percentile(np.abs(columns["raw"]), 99.5)) or 1.0
        ks_spacing = 1.2 * float(np.percentile(np.abs(columns["mask + KS4-like preprocessing"]), 99.5)) or 1.0
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
        for ax, (name, x), spacing in zip(axes, columns.items(), (raw_spacing, raw_spacing, ks_spacing)):
            plot_stacked(ax, t_ms, x, spacing, f"{name}  (spacing {spacing:.0f} uV)")
            ax.axvspan((event["mask_start"] - event["start_sample"]) / fs * 1000,
                       (event["mask_end"] - event["start_sample"]) / fs * 1000, color="r", alpha=0.12)
        fig.suptitle(f"{probe_name} {role}  [{label}]  event at {event['start_sample'] / fs:.4f} s, "
                     f"amp {event['amp_mad']:.1f} MAD, slope {event['slope_mad']:.1f}, "
                     f"peak fraction {event['peak_fraction']:.2f}, duration {event['duration_ms']:.2f} ms, "
                     f"channels {', '.join(map(str, shown_ids))}")
        fig.tight_layout()
        plt.show()


In [ ]:
# --- SpikeInterface: per session  laser mask -> noise mask -> phase shift,  then concatenate + write int16 ---
processed_session.mkdir(parents=True, exist_ok=True)

# job kwargs for write_binary_recording — set n_jobs to 1 if multiprocessing misbehaves
write_job_kwargs = {"n_jobs": 4, "chunk_duration": "2s", "progress_bar": True}

session_boundaries = {}   # probe_name -> list of per-session {sample_start, sample_end, ...}
laser_events = []

for probe_name in probe_names:
    order           = concatenate_order[probe_name]        # e.g. ["behavior", "optotagging"]
    phase_shift_cfg = phase_shift_params[probe_name]
    n_chan          = sessions[order[0]]["probes"][probe_name]["n_chan"]
    sample_shifts   = npx2_sample_shifts(n_chan)            # NP2.0 inter-sample shift, in samples

    per_session_recordings = []
    session_bounds = []
    cursor = 0
    for role in order:
        probe_info = sessions[role]["probes"][probe_name]

        # raw int16 (read-only) -> laser mask (optotagging) -> common-mode noise mask
        recording, n_laser_pulses = build_session_recording(probe_name, role)
        if n_laser_pulses:
            print(f"{probe_name} {role}: masked {n_laser_pulses} laser onsets + {n_laser_pulses} offsets")
        events = noise_events[(probe_name, role)]
        recording = apply_noise_mask(recording, events)
        print(f"{probe_name} {role}: masked {len(events)} common-mode noise events")

        if phase_shift_cfg["apply"]:
            recording = phase_shift(recording,
                                    margin_ms=phase_shift_cfg.get("margin_ms", 40.0),
                                    inter_sample_shift=sample_shifts)

        per_session_recordings.append(recording)
        session_bounds.append({
            "session_id": role, "source_session": sessions[role]["folder"].name,
            "source_dat_bytes": (probe_info["stream_dir"] / "continuous.dat").stat().st_size,
            "sample_start": cursor, "sample_end": cursor + probe_info["n_samples"],
            "n_samples": probe_info["n_samples"]})
        cursor += probe_info["n_samples"]

    # sample-based concatenation (ignore_times=True): no wall-clock alignment across sessions
    full_recording = concatenate_recordings(per_session_recordings)
    output_dat = processed_session / probe_name / "continuous.dat"
    output_dat.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {probe_name}  ->  {output_dat}  ({full_recording.get_num_samples()} samples) ===")
    write_binary_recording(full_recording, output_dat, dtype="int16",
                           add_file_extension=False, **write_job_kwargs)
    session_boundaries[probe_name] = session_bounds

    # every masked pulse, as a sample index into the concatenated axis
    session_start = {entry["session_id"]: entry["sample_start"] for entry in session_bounds}
    if "optotagging" in order:
        probe_info = sessions["optotagging"]["probes"][probe_name]
        probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
        optotagging_start = session_start["optotagging"]
        for laser_name in laser_channels:
            result = opto_results[(probe_name, laser_name)]
            lag_seconds = result["lag_ms"] / 1000.0
            for onset_time, offset_time in zip(result["onset_times"], result["offset_times"]):
                onset_sample  = int(np.searchsorted(probe_timestamps, onset_time + lag_seconds))
                offset_sample = int(np.searchsorted(probe_timestamps, offset_time + lag_seconds))
                laser_events.append({
                    "probe": probe_name, "session": "optotagging", "color": laser_name,
                    "on_sample_concat": onset_sample + optotagging_start,
                    "off_sample_concat": offset_sample + optotagging_start,
                    "on_t_oe": float(onset_time), "lag_ms": float(result["lag_ms"])})

print("\nconcatenation done")

In [ ]:
# --- sidecars: session boundaries, preprocessing provenance, laser events ---
boundaries_doc = {"session_name": session_name,
                  "engine": f"spikeinterface {si.__version__}",
                  "concatenate_order": concatenate_order[probe_names[0]],
                  "streams": {}}
for probe_name in probe_names:
    reference_info = sessions[session_roles[0]]["probes"][probe_name]
    boundaries_doc["streams"][probe_name] = {
        "fs": reference_info["fs"], "n_chan": reference_info["n_chan"],
        "n_samples_total": sum(entry["n_samples"] for entry in session_boundaries[probe_name]),
        "sessions": session_boundaries[probe_name]}
(processed_session / "session_boundaries.json").write_text(json.dumps(boundaries_doc, indent=2))

provenance = {"session_name": session_name, "metadata": str(metadata_path),
              "engine": f"spikeinterface {si.__version__}",
              "sessions": {role: sessions[role]["folder"].name for role in session_roles},
              "probes": {}}
for probe_name in probe_names:
    provenance["probes"][probe_name] = {
        "phase_shift": phase_shift_params[probe_name],
        "mask_laser_artifact": mask_laser_params[probe_name],
        "car": car_params.get(probe_name),
        "mask_noise_artifact": mask_noise_params[probe_name],
        "noise_events": {
            role: {"n_candidates": int(len(noise_candidates[(probe_name, role)])),
                   "n_events": int(len(noise_events[(probe_name, role)])),
                   "masked_fraction": float((noise_events[(probe_name, role)]["mask_end"]
                                             - noise_events[(probe_name, role)]["mask_start"] + 1).sum()
                                            / sessions[role]["probes"][probe_name]["n_samples"])}
            for role in concatenate_order[probe_name]},
        "kilosort_params": config[probe_name]["kilosort_params"],
        "lag_ms": {laser_name: opto_results[(probe_name, laser_name)]["lag_ms"]
                   for laser_name in laser_channels},
        "artifact_channel": {laser_name: {key: opto_results[(probe_name, laser_name)]["artifact"][key]
                                          for key in ("channel", "source", "artifact_sign", "t_value")}
                             for laser_name in laser_channels},
    }
(processed_session / "preprocessing.json").write_text(json.dumps(provenance, indent=2, default=str))
# the metadata exactly as it was read at the start of this run (not re-read from disk)
(processed_session / metadata_path.name).write_text(metadata_text, encoding="utf-8")

if laser_events:
    with open(processed_session / "laser_events.csv", "w", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, list(laser_events[0]))
        writer.writeheader()
        writer.writerows(laser_events)

noise_rows = []
for probe_name in probe_names:
    session_start = {entry["session_id"]: entry["sample_start"] for entry in session_boundaries[probe_name]}
    for role in concatenate_order[probe_name]:
        events = noise_events[(probe_name, role)].copy()
        events.insert(0, "session", role)
        events.insert(0, "probe", probe_name)
        events["mask_start_concat"] = events["mask_start"] + session_start[role]
        events["mask_end_concat"] = events["mask_end"] + session_start[role]
        noise_rows.append(events)
pd.concat(noise_rows, ignore_index=True).to_csv(processed_session / "noise_events.csv", index=False)

print(f"wrote session_boundaries.json, preprocessing.json, noise_events.csv, {metadata_path.name}"
      + (", laser_events.csv" if laser_events else ""))

In [ ]:
# --- verification ---
probe_name = probe_names[0]
n_chan = sessions[session_roles[0]]["probes"][probe_name]["n_chan"]
output = np.memmap(processed_session / probe_name / "continuous.dat", dtype="int16",
                   mode="r").reshape(-1, n_chan)

total_samples = boundaries_doc["streams"][probe_name]["n_samples_total"]
assert output.shape[0] == total_samples, (output.shape, total_samples)
assert (processed_session / probe_name / "continuous.dat").stat().st_size == sum(
    entry["source_dat_bytes"] for entry in session_boundaries[probe_name])
print(f"{probe_name}: {output.shape[0]} samples == sum of sessions, byte count matches sources  OK")

# masked pulse: raw source vs concat output around the first laser event
if laser_events:
    event = laser_events[0]
    probe_info = sessions["optotagging"]["probes"][probe_name]
    probe_channel = lag_estimation_cfg["probe_channel"]
    source = np.memmap(probe_info["stream_dir"] / "continuous.dat", dtype="int16",
                       mode="r").reshape(probe_info["n_samples"], probe_info["n_chan"])
    optotagging_start = {entry["session_id"]: entry["sample_start"]
                         for entry in session_boundaries[probe_name]}["optotagging"]
    source_onset = event["on_sample_concat"] - optotagging_start
    half_window = int(0.006 * probe_info["fs"])

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(np.arange(-half_window, half_window),
            source[source_onset - half_window:source_onset + half_window, probe_channel],
            "k", lw=0.8, label="raw source")
    ax.plot(np.arange(-half_window, half_window),
            output[event["on_sample_concat"] - half_window:event["on_sample_concat"] + half_window, probe_channel],
            "r", lw=0.9, label="concat output")
    ax.axvline(0, color="b", lw=0.7)
    ax.set(title=f"{event['session']} {event['color']} pulse — {probe_name} ch{probe_channel}",
           xlabel="samples from lag-corrected onset", ylabel="int16")
    ax.legend()
    plt.show()

# phase shift moved a quiet channel of the first session by < 1 sample
first_bounds = session_boundaries[probe_name][0]
first_info = sessions[session_roles[0]]["probes"][probe_name]
first_source = np.memmap(first_info["stream_dir"] / "continuous.dat", dtype="int16",
                         mode="r").reshape(first_info["n_samples"], first_info["n_chan"])
window = slice(first_info["n_samples"] // 2, first_info["n_samples"] // 2 + 16000)
source_trace = first_source[window, 5].astype("float64")
output_trace = output[first_bounds["sample_start"] + np.arange(window.start, window.stop), 5].astype("float64")
cross_corr = np.fft.irfft(np.fft.rfft(source_trace - source_trace.mean())
                          * np.conj(np.fft.rfft(output_trace - output_trace.mean())), n=len(source_trace))
peak = int(np.argmax(cross_corr))
peak = peak - len(source_trace) if peak > len(source_trace) // 2 else peak
print(f"phase-shift lag on {probe_name} ch5 (first session): {peak} sample(s)  "
      f"{'OK' if abs(peak) <= 1 else 'CHECK'}")

## Next steps

- **Channel map** (`2.2_generate_channel_map.ipynb`): writes
  `<processed_session>/<probe>/<session_name>_<ProbeX>.mat` from the metadata bad channels.
- **Kilosort** (`2.3_run_kilosort.ipynb`): point `dat_file` at
  `<processed_session>/<probe>/continuous.dat`, set `n_chan_bin` to that probe's
  channel count, set `do_CAR` from `config[<probe>]["kilosort_params"]["do_car"]`.
- **Split back per session** — spike times from the combined sort are sample
  indices into the concatenated axis. Use `session_boundaries.json` to slice them
  back: `sorting.frame_slice(sample_start, sample_end)` for the behavior session
  (first, so `sample_start = 0`). Build a behavior-scoped `analyzer.zarr` from that
  for `jdb_to_nwb` — its `add_kilosort_bombcell_spikes` checks
  `analyzer.get_num_samples() == ephys_num_samples` (the behavior session's full
  sample count), which a concatenated analyzer would fail.
- **Optotagging** — `laser_events.csv` has every pulse as a concatenated-axis
  sample index (subtract the optotagging session's `sample_start` for
  session-local indices).

### SpikeInterface-native alternative for the split-back

Instead of `concatenate_recordings` + manual `session_boundaries.json`, you can
`append_recordings(per_session_recordings)` to build a **multi-segment**
recording. `run_sorter` concatenates the segments internally, and the returned
`sorting` keeps the segments, so `sorting.select_segments([0])` gives the
behavior-session spike train directly (no frame offset). This notebook uses
`concatenate_recordings`, so Kilosort reads a single continuous binary and
`session_boundaries.json` records where each session starts and ends.